In [27]:
import torch.nn as nn
import numpy as np
import torch
import pandas as pd



class NN(nn.Module):
    def __init__(self, n_in, n_out):
        
        super(NN, self).__init__()
        self.act = nn.ReLU()
        self.fc1 = nn.Linear(n_in, 2 * n_in)
        self.fc2 = nn.Linear(2 * n_in, 3 * n_in)
        self.fc3 = nn.Linear(3 * n_in, n_out)        
    
    def forward(self, x):
        
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        x = self.act(x)
        x = self.fc3(x)
        
        return x
    


# Computation of the test statistic
#
# X -- array of univariate observations
#
# p -- positive integer (used for basis construction)
#
def compute_test_stat_nn(X, threshold, t_min=20, n_out_min=10, B=10, delta_max=50, n_epochs=200, model=NN):
    
    #X = X.reshape(-1, 1)
    
    # Sample size
    n = X.shape[0]
    
    # Initialization
    T = np.zeros((n, n))
    
    stopping_time = -1
    
    for t in range(t_min, n):
    
        #if (t % 100) == 0:
        #    print('Iteration', t)
            
        for tau in range(np.maximum(t - n_out_min - delta_max, n_out_min), t-n_out_min):
            
            # Initialize neural network
            f = model(n_in=X.shape[1], n_out=1)
            
            # Parameters of the optimizer
            opt = torch.optim.Adam(f.parameters(), lr=1e-1)
            
            X_t = torch.tensor(X[:t, :], dtype=torch.float32, requires_grad=True)
            
            # weights
            W = torch.cat((torch.ones(tau) * (t - tau), torch.ones(t - tau) * tau)).reshape(-1, 1)
            
            # Create "virtual" labels
            Y_t = torch.cat((torch.ones(tau), torch.zeros(t - tau))).reshape(-1, 1)
    
            # Loss function    
            loss_fn = nn.BCEWithLogitsLoss(weight=W)
            
            # Neural network training
            for epoch in range(n_epochs):
                
                loss = loss_fn(f(X_t), Y_t).mean()
                loss.backward()
                opt.step()
                opt.zero_grad()
                
            Z = f(X_t).detach().numpy().reshape(-1)
            
            # Use thresholding to avoid numerical issues
            Z = np.minimum(Z, B)
            Z = np.maximum(Z, -B)
            
            D = np.zeros(t)
            D[:tau] = 2 / (1 + np.exp(-Z[:tau]))
            D[tau:] = 2 / (1 + np.exp(Z[tau:]))
            D = np.log(D)
            
            # Compute statistics for each t
            # and each change point candidate tau
            T[tau, t] = tau * (t - tau) / t * (np.mean(D[:tau]) + np.mean(D[tau:]))
            
        if (np.max(T[:, t]) > threshold):
            
            stopping_time = t
            break
       
    # Array of test statistics
    S = np.max(T[:, :stopping_time + 1], axis=0)
    
    return S, stopping_time


X = pd.read_csv("../data/HAR_NEW/data1.csv").values
change_points = np.load("../data/HAR_NEW/data1.npy")


change_points

array([ 28,  52,  79, 126, 151, 177, 203, 214, 226, 238, 249, 297, 304,
       331])

In [ ]:
# Initialization
st_nn = 0
new_st_nn = 0

# the threshold
z_nn = 1

# Initialization of the test statistic
S_nn = np.empty(0)

# Initialization of the list of detected change points
change_points_nn = []


# Initialization of the delays array and
# the false alarms counter
delays_nn = np.empty(0)
current_change_point_ind = 0
false_alarms_nn = 0

data = X.copy()

while new_st_nn >= 0:
    
    # Run the procedure until the moment
    # it reports a change point occurrence
    new_S_nn, new_st_nn = compute_test_stat_nn(data[st_nn + 1:], z_nn, n_epochs=20)
    
    S_nn = np.append(S_nn, new_S_nn)
    
    st_nn += new_st_nn
    change_points_nn += [int(st_nn)]
    
    if(new_st_nn > 0):
        print('Detected change point (neural networks):', st_nn)
        
    if (st_nn < change_points[current_change_point_ind]):
        false_alarms_nn += 1
    
    else:
        delays_nn = np.append(delays_nn, np.array([st_nn - change_points[current_change_point_ind]]), axis=0)
        current_change_point_ind += 1

print('Neural networks. Number of false alarms:', false_alarms_nn, '; average delay:', np.mean(delays_nn),\
     '±', np.std(delays_nn))

Detected change point (neural networks): 39
Detected change point (neural networks): 71
Detected change point (neural networks): 93
Detected change point (neural networks): 124
Detected change point (neural networks): 154
Detected change point (neural networks): 179
Detected change point (neural networks): 241
Detected change point (neural networks): 262
Detected change point (neural networks): 304
Detected change point (neural networks): 332
Neural networks. Number of false alarms: 1 ; average delay: 51.2 ± 34.04937591204867


In [1]:
import wrappers
import test_utils
from test_utils import Tester, run_test_on_dataset


In [4]:
%%time

dir_path = "../data/HAR_NEW/"


models = [wrappers.Contrastive_NN(threshold=1, t_min=20, n_out_min=10, B=10, delta_max=50, n_epochs=10)]
          
          #wrappers.OrigRuLSIF(alpha=0.1, kernel_num=10, window_size=100, step=10, periods=1, height=0.1)]


for amodel in models:
    report = run_test_on_dataset(dir_path, amodel, downsample=1, margin=20, verbose=0, sigma=0.0)
    print(amodel)
    for rep in report:
        display(rep)

[ 28  52  79 126 151 177 203 214 226 238 249 297 304 331]
[28, 52, 79, 126, 151, 177, 203, 214, 226, 238, 249, 297, 304, 331, 347]
[30, 56, 77, 100, 123, 144, 165, 186, 209, 234, 256, 283, 304, 325, 324]
[ 24  48  79 106 125 148 169 199 226 252 271]
[24, 48, 79, 106, 125, 148, 169, 199, 226, 252, 271, 294]
[23, 48, 70, 92, 113, 134, 155, 178, 202, 225, 246, 271, 270]
[ 22  48  78 107 129 155 181 208 235 265 289]
[22, 48, 78, 107, 129, 155, 181, 208, 235, 265, 289, 316]
[23, 45, 77, 99, 120, 141, 162, 186, 209, 234, 258, 279, 300, 299]
[ 28  56  83 110 135 162 196 219 252 275 296]
[28, 56, 83, 110, 135, 162, 196, 219, 252, 275, 296, 320]
[23, 45, 66, 87, 114, 137, 159, 180, 201, 223, 250, 279, 303, 302]
[ 28  52  85 115 142 173 203 228 257 284 304]
[28, 52, 85, 115, 142, 173, 203, 228, 257, 284, 304, 327]
[24, 46, 68, 91, 113, 138, 159, 181, 210, 232, 255, 278, 302, 325, 324]
[ 31  57  82 112 136 163 193 221 247 276 297]
[31, 57, 82, 112, 136, 163, 193, 221, 247, 276, 297, 323]
[27, 51,

,0,1,2,3
F1,0.838492,0.083703,0.666667,0.960000
AUC,0.846936,0.083462,0.667279,0.961538
FP,3.433333,1.633345,1.000000,7.000000
DD,6.459123,1.191855,4.428571,9.222222
Threshold,10.000000,0.000000,10.000000,10.000000
Covering,0.649862,0.052342,0.513629,0.739708
Runtime,17.634074,6.205876,8.120794,34.378712


,0,1,2,3
F1,0.857854,0.064293,0.705882,0.960000
AUC,0.866532,0.064043,0.708333,0.961538
FP,0.333333,0.711159,0.000000,3.000000
DD,6.459123,1.191855,4.428571,9.222222
Threshold,20.000000,0.000000,20.000000,20.000000
Covering,0.649862,0.052342,0.513629,0.739708
Runtime,17.634074,6.205876,8.120794,34.378712


,0,1,2,3
F1,0.868362,0.052921,0.764706,0.960000
AUC,0.877089,0.051557,0.767361,0.961538
FP,0.000000,0.000000,0.000000,0.000000
DD,6.459123,1.191855,4.428571,9.222222
Threshold,30.000000,0.000000,30.000000,30.000000
Covering,0.649862,0.052342,0.513629,0.739708
Runtime,17.634074,6.205876,8.120794,34.378712


,0,1,2,3
F1,0.873739,0.049023,0.764706,0.960000
AUC,0.882567,0.047451,0.767361,0.961538
FP,0.000000,0.000000,0.000000,0.000000
DD,6.459123,1.191855,4.428571,9.222222
Threshold,40.000000,0.000000,40.000000,40.000000
Covering,0.649862,0.052342,0.513629,0.739708
Runtime,17.634074,6.205876,8.120794,34.378712


,0,1,2,3
F1,0.878112,0.049918,0.764706,0.960000
AUC,0.886960,0.047923,0.767361,0.961538
FP,0.000000,0.000000,0.000000,0.000000
DD,6.459123,1.191855,4.428571,9.222222
Threshold,50.000000,0.000000,50.000000,50.000000
Covering,0.649862,0.052342,0.513629,0.739708
Runtime,17.634074,6.205876,8.120794,34.378712


CPU times: user 2h 10s, sys: 9min 59s, total: 2h 10min 9s
Wall time: 8min 50s


In [7]:
%%time

dir_path = "../data/MNIST_SWCPD/"


models = [wrappers.Contrastive_NN(threshold=15, t_min=100, n_out_min=10, B=10, delta_max=50, n_epochs=5)]
          
          #wrappers.OrigRuLSIF(alpha=0.1, kernel_num=10, window_size=100, step=10, periods=1, height=0.1)]


for amodel in models:
    report = run_test_on_dataset(dir_path, amodel, downsample=1, margin=20, verbose=0, sigma=0.0)
    print(amodel)
    for rep in report:
        display(rep)

[200 400]
[200, 400, 600]


KeyboardInterrupt: 